<a href="https://colab.research.google.com/github/lenmecc/miniature-enigma/blob/main/v10_bottlenecks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# HERRAMIENTA: ANALISIS DE CUELLOS DE BOTELLA + MULTI-CALIBRE
# AUTOR: LFRG & Balde
# VERSION: 13.0 (Adaptado a Mix de Calibres Decapados)
# ==========================================

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import io
from google.colab import files

# --- CONFIGURACION MAESTRA DE PLANTA ---
CONFIG_PLANTA = {
    '02_Corte':      {'capacidad': 62.0,  'efic': 0.56, 'unidad': 'Hrs'},
    '02_Doblez':     {'capacidad': 122,   'efic': 0.47, 'unidad': 'Hrs'},
    '02_Soldadura':  {'capacidad': 220,   'efic': 0.54, 'unidad': 'Hrs'},
    '02_Pintura':    {'capacidad': 200.0, 'efic': 0.95, 'unidad': 'Mts'},
    '02_Op Secundarias': {'capacidad': 65,   'efic': 0.73, 'unidad': 'Hrs'},
    "02_Nivelado":   {'capacidad': 8.6,   'efic': 0.33, 'unidad': 'Hrs'},
}

DEFAULT_CONFIG = {'capacidad': 19.0, 'efic': 0.85, 'unidad': 'Hrs'}
PROCESOS_LOGISTICOS = ["05_Proceso Externo", "02_Embarques" , "02_Almacen wip"]
META_DIAS_MAXIMO = 5.0

# --- DATOS FINANCIEROS (Mix de Calibres Decapados) ---
# El precio por Kg del acero decapado es relativamente estable entre calibres.
COSTO_USD_KG = 1.10

# ==========================================
# MODULO DE INSIGHTS (MULTI-CALIBRE)
# ==========================================
def generar_insights_estrategicos(df_grouped):
    print("\n" + "█"*70)
    print("🧠 EXECUTIVE INSIGHTS (Mix de Calibres - Acero Decapado)")
    print("█"*70)

    # 1. VALUACION FINANCIERA (La verdad universal: El Peso)
    total_wip_kg = df_grouped['RW'].sum()
    valor_total_usd = total_wip_kg * COSTO_USD_KG

    wip_logistico = df_grouped[df_grouped['Proceso'].isin(PROCESOS_LOGISTICOS)]['RW'].sum()
    valor_logistico = wip_logistico * COSTO_USD_KG
    valor_operativo = valor_total_usd - valor_logistico

    print(f"\n💰 1. ESTADO DE RESULTADOS (Valuación por Peso @ ${COSTO_USD_KG} USD/kg):")
    print(f"   - Valor Total Inventario:      ${valor_total_usd:,.2f} USD")
    print(f"   -------------------------------------------------------------")
    print(f"   - Capital Activo (Máquinas):   ${valor_operativo:,.2f} USD")
    print(f"   - Capital Pasivo (Logística):  ${valor_logistico:,.2f} USD ({(valor_logistico/valor_total_usd)*100:.1f}%)")

    if valor_logistico > 50000:
        print(f"   ⚠️ ALERTA: Tienes ${valor_logistico/1000:,.0f}k USD detenidos en logística/externos.")

    # 2. ANALISIS DE PARETO
    df_pareto = df_grouped.sort_values(by='RW', ascending=False).copy()
    top_3 = df_pareto.head(3)
    nombres_top = ", ".join(top_3['Proceso'].tolist())

    print(f"\n📊 2. CONCENTRACIÓN (Pareto):")
    print(f"   - Tus mayores 'bancos' de dinero son: {nombres_top}")
    print(f"   - Solo el {top_3['RW'].sum()/total_wip_kg*100:.1f}% del peso total está ahí.")

    # 3. CONVERSION TECNICA (Rango Estimado)
    # Al tener varios calibres, damos un rango de referencia para validación
    print(f"\n📐 3. FACTORES TÉCNICOS (Referencia para Pintura):")
    print(f"   Dado que manejas múltiples calibres, la relación m2 vs Kg varía:")
    print(f"   - Calibres Delgados (ej. Cal 14/1.9mm): ~15 kg/m2")
    print(f"   - Calibres Medios   (ej. Cal 10/3.4mm): ~27 kg/m2")
    print(f"   - Calibres Gruesos  (ej. Cal 07/4.5mm): ~36 kg/m2")
    print(f"   *Tip: Si Pintura procesa muchos metros pero el peso (Kg) baja poco,")
    print(f"         están pasando mayormente material delgado (Cal 12-16).*")

    # 4. GAP DE EFICIENCIA
    print(f"\n⚙️ 4. OPORTUNIDADES DE EFICIENCIA (Gap Analysis):")
    for proc, cfg in CONFIG_PLANTA.items():
        if proc in df_grouped['Proceso'].values:
            efic = cfg['efic']
            if efic < 0.60:
                cap_teorica = cfg['capacidad']
                gap = cap_teorica * (0.85 - efic)
                print(f"   - {proc} (Efic actual {efic*100:.0f}%): Subir al 85% liberaría {gap:.1f} {cfg['unidad']}/día.")

    print("\n" + "="*70)

def generar_reporte_final():
    print("Generando analisis de cuellos de botella v13.0...")
    print(">> SUBE TU ARCHIVO")

    uploaded = files.upload()
    if not uploaded: return

    filename = next(iter(uploaded))
    try:
        if filename.endswith('.csv'):
            df = pd.read_csv(io.BytesIO(uploaded[filename]))
        elif filename.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(io.BytesIO(uploaded[filename]))
        else: return
    except: return

    # LIMPIEZA
    df.columns = [c.strip() for c in df.columns]
    df['RW'] = pd.to_numeric(df['RW'], errors='coerce').fillna(0)
    df['Hrs'] = pd.to_numeric(df['Hrs'], errors='coerce').fillna(0)
    df['Proceso'] = df['Proceso'].astype(str).str.strip()

    df_grouped = df.groupby('Proceso')[['RW', 'Hrs']].sum().reset_index()

    # CALCULO KPIs
    def calcular_metricas(row):
        proc = row['Proceso']
        if proc in PROCESOS_LOGISTICOS:
            return pd.Series([0.0, 'Logistica', 0.0])

        cfg = CONFIG_PLANTA.get(proc, DEFAULT_CONFIG)
        cap_real = cfg['capacidad'] * cfg['efic']
        dias = row['Hrs'] / cap_real if cap_real > 0 else 0
        return pd.Series([dias, cfg['unidad'], cap_real])

    df_grouped[['Dias_Atraso', 'Unidad', 'Cap_Real']] = df_grouped.apply(calcular_metricas, axis=1)

    # FILTROS
    df_scatter = df_grouped[
        (~df_grouped['Proceso'].isin(PROCESOS_LOGISTICOS)) &
        (df_grouped['Dias_Atraso'] >= 1.0)
    ].copy()

    df_barras = df_grouped.sort_values(by='RW', ascending=True)
    df_barras = df_barras[df_barras['RW'] > 1000].copy()

    # VISUALIZACION
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.20,
        subplot_titles=("Mapa de Carga Operativa (> 1 Día)", "Inventario Físico Significativo (> 1 Ton)")
    )

    if not df_scatter.empty:
        fig.add_trace(go.Scatter(
            x=df_scatter['Hrs'],
            y=df_scatter['RW'],
            mode='markers+text',
            text=df_scatter['Proceso'],
            textposition="top center",
            name='Procesos Críticos',
            marker=dict(
                size=df_scatter['Hrs'],
                sizemode='area', sizeref=2.*max(df_scatter['Hrs'])/(50.**2), sizemin=8,
                color=df_scatter['Dias_Atraso'], colorscale='Reds',
                colorbar=dict(title="Días Atraso", len=0.5, y=0.8)
            ),
            hovertemplate =
            '<b>%{text}</b><br> Atraso: <b>%{marker.color:.1f} Días</b><br>' +
            ' Capacidad: %{customdata[1]:.1f} %{customdata[0]}/día<br>' +
            ' WIP: %{y:,.0f} kg<extra></extra>',
            customdata=df_scatter[['Unidad', 'Cap_Real']]
        ), row=1, col=1)

    if not df_barras.empty:
        colores = ['#f1c40f' if p in PROCESOS_LOGISTICOS else '#2c3e50' for p in df_barras['Proceso']]

        fig.add_trace(go.Bar(
            x=df_barras['RW'], y=df_barras['Proceso'], orientation='h',
            text=df_barras['RW'].apply(lambda x: f"{x:,.0f} kg"), textposition='auto',
            marker=dict(color=colores),
            showlegend=False
        ), row=2, col=1)

        fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color='#2c3e50'), name='Operaciones'), row=2, col=1)
        fig.add_trace(go.Bar(x=[None], y=[None], marker=dict(color='#f1c40f'), name='Procesos Logísticos'), row=2, col=1)

    fig.update_layout(
        title=f"<b>Tablero de identifiacón de cuellos de botella</b><br><sup>Valuación Inventario @ ${COSTO_USD_KG} USD/kg</sup>",
        template="plotly_white",
        height=950,
        showlegend=True,
        legend=dict(orientation="h", yanchor="top", y=-0.08, xanchor="right", x=1),
        margin=dict(b=120)
    )

    fig.update_yaxes(tickfont=dict(size=9), row=2, col=1)
    fig.show()

    # LLAMADA A INSIGHTS
    generar_insights_estrategicos(df_grouped)

if __name__ == "__main__":
    generar_reporte_final()

Generando analisis de cuellos de botella v13.0...
>> SUBE TU ARCHIVO


Saving Consulta de etiquetas_wk21.xlsx to Consulta de etiquetas_wk21.xlsx



██████████████████████████████████████████████████████████████████████
🧠 EXECUTIVE INSIGHTS (Mix de Calibres - Acero Decapado)
██████████████████████████████████████████████████████████████████████

💰 1. ESTADO DE RESULTADOS (Valuación por Peso @ $1.1 USD/kg):
   - Valor Total Inventario:      $271,600.77 USD
   -------------------------------------------------------------
   - Capital Activo (Máquinas):   $159,630.83 USD
   - Capital Pasivo (Logística):  $111,969.94 USD (41.2%)
   ⚠️ ALERTA: Tienes $112k USD detenidos en logística/externos.

📊 2. CONCENTRACIÓN (Pareto):
   - Tus mayores 'bancos' de dinero son: 02_Soldadura, 02_Almacen wip, 02_Embarques
   - Solo el 60.1% del peso total está ahí.

📐 3. FACTORES TÉCNICOS (Referencia para Pintura):
   Dado que manejas múltiples calibres, la relación m2 vs Kg varía:
   - Calibres Delgados (ej. Cal 14/1.9mm): ~15 kg/m2
   - Calibres Medios   (ej. Cal 10/3.4mm): ~27 kg/m2
   - Calibres Gruesos  (ej. Cal 07/4.5mm): ~36 kg/m2
   *Tip: Si Pin